[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Migrations &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, which also creates `/tmp/guide_migrations` and puts two
documents in `mig_people`. Run it first. Each task calls `fresh_migrations()` and `reset_people()`
before it starts, so they can be run in any order and as often as you like.


In [1]:
import os
import pathlib
import random
import shutil
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

WORK = pathlib.Path("/tmp/guide_migrations")                        # a folder on disk, not in here
STANDALONE = "mongodb://127.0.0.1:27018"                            # the plain mongod, for one error


def fresh_migrations():
    """An empty migrations folder. The names are ours, so the outputs below are reproducible."""
    shutil.rmtree(WORK, ignore_errors=True)
    (WORK / "migrations").mkdir(parents=True)
    return WORK


def write_migration(filename, body):
    """Write one migration file. beanie new-migration would name it after the clock."""
    path = WORK / "migrations" / filename
    path.write_text(body.strip() + "\n")
    return path.name


def run_beanie(*arguments, cwd=None):
    """Run the beanie command as a subprocess and hand back its exit code and last lines.

    It is a subprocess because that is what the command is: it imports your migration files in a
    fresh interpreter, which is why models defined in a notebook cell are invisible to it."""
    done = subprocess.run([sys.executable, "-m", "beanie.executors.migrate", *arguments],
                          cwd=cwd or WORK, capture_output=True, text=True)
    output = (done.stdout + done.stderr).strip().splitlines()
    return done.returncode, output


def people(port=27017):
    with pymongo.MongoClient(f"mongodb://127.0.0.1:{port}/shop") as client:
        shop = client.get_default_database()
        return [dict(sorted(row.items())) for row in
                shop.mig_people.find({}, {"_id": 0}).sort("name")]


def reset_people(port=27017):
    """Two documents and no migration history, so every section starts from the same place."""
    with pymongo.MongoClient(f"mongodb://127.0.0.1:{port}/shop") as client:
        shop = client.get_default_database()
        shop.mig_people.drop()
        shop.migrations_log.drop()                                  # not "migrations": the log
        shop.mig_people.insert_many([{"name": "ana"}, {"name": "bo"}])
        return shop.mig_people.count_documents({})


ADD_GREETING = """
from beanie import Document, iterative_migration


class Person(Document):
    name: str

    class Settings:
        name = "mig_people"


class Greeted(Document):
    name: str
    greeting: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @iterative_migration()
    async def add_greeting(self, input_document: Person, output_document: Greeted):
        output_document.greeting = f"hello {input_document.name}"


class Backward:
    @iterative_migration()
    async def drop_greeting(self, input_document: Greeted, output_document: Person):
        pass
"""


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print("work:   ", fresh_migrations())
print("people: ", reset_people(), "documents")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
work:    /tmp/guide_migrations
people:  2 documents
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** What the command generates.


In [2]:
fresh_migrations()
exit_code, output = run_beanie("new-migration", "-n", "add_nickname", "-p", "migrations")

made = sorted(path.name for path in (WORK / "migrations").glob("*.py"))
print("exit code:", exit_code, "| name ends with:", made[0].split("_", 1)[1])
print("contents:")
print((WORK / "migrations" / made[0]).read_text().strip())


exit code: 0 | name ends with: add_nickname.py
contents:
class Forward: ...


class Backward: ...


Two empty class bodies. The timestamp at the front of the name is what orders the migrations, and
it is why this notebook writes its own filenames instead.


**2.** A field added to every document.


In [3]:
ADD_NICKNAME = """
from beanie import Document, iterative_migration


class Person(Document):
    name: str

    class Settings:
        name = "mig_people"


class Nicknamed(Document):
    name: str
    nickname: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @iterative_migration()
    async def add_nickname(self, input_document: Person, output_document: Nicknamed):
        output_document.nickname = input_document.name.upper()


class Backward: ...
"""

fresh_migrations()
reset_people()
write_migration("20260101000000_add_nickname.py", ADD_NICKNAME)

print("before:", people())
run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017", "-db", "shop", "-p", "migrations")
print("after: ", people())


before: [{'name': 'ana'}, {'name': 'bo'}]
after:  [{'name': 'ana', 'nickname': 'ANA'}, {'name': 'bo', 'nickname': 'BO'}]


`Person` is the shape now and `Nicknamed` is the shape wanted. Both name the same collection, which
is what makes this a change to those documents rather than a copy elsewhere.


**3.** Only once.


In [4]:
exit_code, output = run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017",
                               "-db", "shop", "-p", "migrations")
print("second run said:", output[-1] if output else "(nothing)")

with pymongo.MongoClient(URI) as client:
    print("log:", [row["name"] for row in client.get_default_database().migrations_log.find()])
print("documents:", people())


second run said: Building migration list
log: ['20260101000000_add_nickname.py']
documents: [{'name': 'ana', 'nickname': 'ANA'}, {'name': 'bo', 'nickname': 'BO'}]


The name is in `migrations_log`, so Beanie skips it. Dropping that collection is what makes a
migration runnable again, and it is not something to do to a real database.


**4.** The whole collection, in one operation.


In [5]:
LOWER_IT = """
from beanie import Document, free_fall_migration


class Person(Document):
    name: str
    nickname: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @free_fall_migration(document_models=[Person])
    async def lower_nicknames(self, session):
        await Person.get_pymongo_collection().update_many(
            {}, [{"$set": {"nickname": {"$toLower": "$nickname"}}}], session=session)


class Backward: ...
"""

write_migration("20260102000000_lower_it.py", LOWER_IT)
run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017", "-db", "shop", "-p", "migrations")
print("after the free fall migration:", people())


after the free fall migration: [{'name': 'ana', 'nickname': 'ana'}, {'name': 'bo', 'nickname': 'bo'}]


One `update_many` on the server rather than a read and a write per document. Far faster over a large
collection, and with no validation of what it produces.


**5.** Undoing one.


In [6]:
REVERSIBLE = """
from beanie import Document, iterative_migration


class Before(Document):
    name: str

    class Settings:
        name = "mig_people"


class After(Document):
    name: str
    shouted: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @iterative_migration()
    async def add(self, input_document: Before, output_document: After):
        output_document.shouted = input_document.name.upper()


class Backward:
    @iterative_migration()
    async def remove(self, input_document: After, output_document: Before):
        pass
"""

fresh_migrations()
reset_people()
write_migration("20260101000000_reversible.py", REVERSIBLE)

run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017", "-db", "shop", "-p", "migrations")
print("forward: ", people())

run_beanie("migrate", "--backward", "-d", "1", "-uri", "mongodb://127.0.0.1:27017",
           "-db", "shop", "-p", "migrations")
print("backward:", people())


forward:  [{'name': 'ana', 'shouted': 'ANA'}, {'name': 'bo', 'shouted': 'BO'}]
backward: [{'name': 'ana'}, {'name': 'bo'}]


`Backward` is a migration in its own right, written by you in the opposite direction. Beanie infers
nothing, which is why the generated file leaves it empty and why most projects leave it that way.


**6.** Against a server with no replica set.


In [7]:
fresh_migrations()
reset_people(port=27018)
write_migration("20260101000000_add_greeting.py", ADD_GREETING)

exit_code, output = run_beanie("migrate", "-uri", STANDALONE, "-db", "shop", "-p", "migrations")
print("as it is:", [line for line in output
                    if "OperationFailure" in line][-1].split(", full error")[0])

exit_code, output = run_beanie("migrate", "--no-use-transaction", "-uri", STANDALONE,
                               "-db", "shop", "-p", "migrations")
print("with --no-use-transaction:", people(port=27018))

shutil.rmtree(WORK, ignore_errors=True)
for port in (27017, 27018):
    with pymongo.MongoClient(f"mongodb://127.0.0.1:{port}/shop") as client:
        client.get_default_database().mig_people.drop()
        client.get_default_database().migrations_log.drop()


as it is: pymongo.errors.OperationFailure: Transaction numbers are only allowed on a replica set member or mongos
with --no-use-transaction: [{'greeting': 'hello ana', 'name': 'ana'}, {'greeting': 'hello bo', 'name': 'bo'}]


The migration runs the whole way in a transaction, and a transaction needs a replica set.
`--no-use-transaction` gets the work done and gives up the guarantee that a failure halfway leaves
nothing behind.


---

&#8592; **Back to:** [Migrations](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/15-migrations.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
